# **Rastreando múltiplos pokémons em vídeos**

## **Preparação das imagens**

Primeiramente, devemos importar bibliotecas com funcionalidades úteis ou que que pré implementam partes importantes para o programa.

In [117]:
import cv2 as cv
import numpy as np
import glob
import os

Salvando os caminhos para os arquivos:

In [118]:
VIDEO_PATH = '/content/drive/MyDrive/IC/labs_profis/multi_pokemons_videos/pseudovideo_multi_2.mp4'
TEMPLATES_DIR = '/content/drive/MyDrive/IC/labs_profis/images/1st_gen'
OUT_VIDEO = '/content/tracked_output_4.mp4'

THRESHOLD    = 0.70

Vamos começar montando uma lista com todos os arquivos .png que estão dentro da pasta TEMPLATES_DIR. ```glob.glob()``` percorre o sistema de arquivos e retorna todos aqueles terminados em .png dentro da pasta. ```template_paths``` retorna uma lista de strings com os caminhos completos.

In [119]:
pattern = TEMPLATES_DIR + "/*.png"
template_paths = glob.glob(pattern)

templates = []
for path in template_paths:
    t_gray = cv.imread(path, cv.IMREAD_GRAYSCALE)
    tw, th = t_gray.shape[::-1]
    templates.append((t_gray, tw, th)) # precisamos retornar uma tupla pois queremos essas 3 infos para cada um dos templates

## **Preparação do vídeo**

In [120]:
cap = cv.VideoCapture(VIDEO_PATH)

fps = cap.get(cv.CAP_PROP_FPS)
if fps is None or fps <= 0 or np.isnan(fps):
    fps = 30.0

width  = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

fourcc = cv.VideoWriter_fourcc(*"mp4v")
writer = cv.VideoWriter(OUT_VIDEO, fourcc, fps, (width, height))

## **Performando o rastreamento pokémon**

### Contando os Pokémons

Primeiramente podemos contar quantos pokémons há na cena

In [121]:
count = 0

ok, frame = cap.read()
gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

for t_gray, tw, th in templates:
    res = cv.matchTemplate(gray, t_gray, cv.TM_CCOEFF_NORMED)
    loc = np.where(res >= THRESHOLD)
    res_row = loc [0]
    res_column = loc [1]

    count += len(loc[0])  # conta todas as ocorrências

In [122]:
print(f"Foram encontrados {count} pokémons.")

Foram encontrados 3 pokémons.


### Encontrando os Pókémons

Agora vamos iterar sobre os frames com o pokémon se movendo pelo cenário, sobrepondo janelas (patches) w x h do fundo sobre os templates e comparando o resultado conforme o método escolhido.

In [123]:
while True:
    ok, frame = cap.read()
    if not ok:
        break

    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    for t_gray, tw, th in templates:
        res = cv.matchTemplate(gray, t_gray, cv.TM_CCOEFF_NORMED)
        res_row, res_column = np.where(res >= THRESHOLD) # par (res_row, res_column) é um candidato de posição detectada

        match_coords = list(zip(res_row, res_column))

        for top_left in match_coords:
            cv.rectangle(frame, (top_left[1], top_left[0]), (top_left[1] + tw, top_left[0] + th), (0, 0, 255), 2)
            # (top_left_x + tw, top_left_y + th) => canto inferior direito
            # (top_left_x, top_left_y) => canto superior esquerdo

            #top_left[0]=y, top_left[1]=x

    writer.write(frame)


cap.release()
writer.release()

No caso, usamos o método zip para iterar sobre as coordenadas onde houve match pois ```res_row``` tem todos os índices de linha (coordenada y) onde a condição foi satisfeita e ```res_column``` tem todos os índices de coluna (coordenada x) correspondentes. Ex.: res_row = [10, 20, 30] e res_column = [40, 50, 60]. Significa que foram encontrados matches em (10,40), (20,50) e (30,60).

## **Exibindo o vídeo do resultado no Colab**

Como o cv.imshow não funciona aqui, vamos precisar usar um método alternativo para visualizar o vídeo rastreado sem precisar baixá-lo no computador.

Para isso, vamos convertê-lo para um tipo de arquivo compatível com o Colab usando a biblioteca ```imageio``` e depois exibir o vídeo com o auxilio de outra biblioteca: ```IPython```.

In [124]:
import imageio.v2 as imageio
from IPython.display import Video, display

In [125]:
reader = imageio.get_reader(OUT_VIDEO)
writer = imageio.get_writer(OUT_VIDEO, fps=reader.get_meta_data().get("fps", fps), codec="libx264", quality=8)
for frame in reader:
    writer.append_data(frame)
writer.close()

display(Video(OUT_VIDEO, embed=True, html_attributes="controls autoplay loop"))